<a href="https://colab.research.google.com/github/emmex43/Machine-learning-fantasy/blob/main/Copy_of_Aeroguard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 45.9 MB/s eta 0:00:00


In [ ]:
from roboflow import Roboflow
import os, shutil

# Dataset 1 - wwj
rf = Roboflow(api_key="Get yours on Roboflow")
project = rf.workspace("wwj-bjihs").project("oil-spill-ed2aj")
version = project.version(1)
dataset1 = version.download("yolov8")

# Dataset 2 - khalid
rf = Roboflow(api_key="Get yours on Roboflow")
project2 = rf.workspace("khalid-alhilali").project("oil-spill-eojff")
version2 = project2.version(1)
dataset2 = version2.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to oil-spill-1 in yolov8:: 100%|██████████| 518/518 [00:00<00:00, 5635.02it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...


In [ ]:
import os, shutil

base = "/content/merged"
for split in ["train", "valid", "test"]:
    os.makedirs(f"{base}/{split}/images", exist_ok=True)
    os.makedirs(f"{base}/{split}/labels", exist_ok=True)

def merge(src, split, prefix):
    for folder in ["images", "labels"]:
        src_path = f"{src}/{split}/{folder}"
        dst_path = f"{base}/{split}/{folder}"
        if not os.path.exists(src_path):
            continue
        for f in os.listdir(src_path):
            ext = os.path.splitext(f)[1]
            new_name = f"{prefix}_{f}"
            shutil.copy(f"{src_path}/{f}", f"{dst_path}/{new_name}")

# Clear and redo
shutil.rmtree(base)
for split in ["train", "valid", "test"]:
    os.makedirs(f"{base}/{split}/images", exist_ok=True)
    os.makedirs(f"{base}/{split}/labels", exist_ok=True)

for i, ds in enumerate([dataset1.location, dataset2.location]):
    for split in ["train", "valid", "test"]:
        merge(ds, split, prefix=f"ds{i+1}")

print("Merged dataset counts:")
for split in ["train", "valid", "test"]:
    n = len(os.listdir(f"{base}/{split}/images"))
    print(f"  {split}: {n} images")

Merged dataset counts:
  train: 354 images
  valid: 102 images
  test: 50 images


In [ ]:
import os, glob

def fix_labels(base_dir):
    for split in ["train", "valid", "test"]:
        label_dir = f"{base_dir}/{split}/labels"
        if not os.path.exists(label_dir):
            continue
        for label_file in glob.glob(f"{label_dir}/*.txt"):
            with open(label_file, 'r') as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'  # remap all classes to oil_spill
                    new_lines.append(' '.join(parts) + '\n')
            with open(label_file, 'w') as f:
                f.writelines(new_lines)

fix_labels("/content/merged")
print("All labels remapped to class 0")

All labels remapped to class 0


In [ ]:
yaml_content = f"""
path: {base}
train: train/images
val: valid/images
test: test/images

nc: 1
names: ['oil_spill']
"""
with open(f"{base}/data.yaml", "w") as f:
    f.write(yaml_content)
print("data.yaml created")

data.yaml created


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano = fastest, good for hackathon
results = model.train(
    data=f"{base}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="aeroguard_v1",
    patience=10
)

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aeroguard_v1-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

In [ ]:
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

# Save best weights
shutil.copy(
    "/content/runs/detect/aeroguard_v1/weights/best.pt",
    "/content/aeroguard_best.pt"
)
print("Model saved to /content/aeroguard_best.pt")

Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1520.3±453.3 MB/s, size: 52.5 KB)
val: Scanning /content/merged/valid/labels.cache... 102 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 102/102 38.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.5it/s 2.8s
                   all        102        238      0.783      0.798      0.827       0.62
Speed: 4.6ms preprocess, 6.1ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /content/runs/detect/val-2
mAP50: 0.827
Precision: 0.783
Recall: 0.798
Model saved to /content/aeroguard_best.pt
